# Brasileirão – probabilidades H/D/A (LogLoss)
Ideia: as cotações só existem no treino, mas carregam muito mais sinal que o resultado (ruidoso).
Usamos as probabilidades implícitas nas cotações (sem margem) como **alvo suave** (destilação):
uma Ridge multivariada prevê os log-odds (A/D, H/D) a partir das features da linha + identidade dos times
(dummies mandante/visitante). Nenhuma cotação é usada como feature; no teste só a linha da própria partida é usada.
Validação: janela expansiva por temporada (2016–2021), sem shuffle.

In [1]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')
from scipy.special import softmax
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
tr=pd.read_csv('data/train.csv'); te=pd.read_csv('data/test.csv')
CLS=['A','D','H']  # ordem alfabética = ordem do log_loss do sklearn e da submissão

In [2]:
def fe(d):
    d=d.copy()
    d['ed2']=d.elo_diff.abs()
    d['fp']=d.form_pts_home-d.form_pts_away
    d['fgd']=(d.form_gf_home-d.form_ga_home)-(d.form_gf_away-d.form_ga_away)
    d['hv']=d.home_form_as_host-d.away_form_as_visitor
    d['rest']=np.log1p(d.rest_days_home.clip(0,30))-np.log1p(d.rest_days_away.clip(0,30))
    d['ppg']=d.season_ppg_home-d.season_ppg_away
    return d
tr,te=fe(tr),fe(te)
NUM=['elo_diff','ed2','fp','fgd','hv','ppg','rest','h2h_home_pts','form_pts_home','form_pts_away']
TEAMS=sorted(set(tr.Home)|set(tr.Away)|set(te.Home)|set(te.Away))
def X(d):
    x=d[NUM].copy()
    for t in TEAMS: x['h_'+t]=(d.Home==t).astype(float); x['a_'+t]=(d.Away==t).astype(float)
    return x
# alvo suave: probabilidades implícitas das cotações, normalizadas (remove margem)
inv=1/tr[['odd_visitante','odd_empate','odd_casa']].values
P=inv/inv.sum(1,keepdims=True)
T=np.log(P)-np.log(P[:,[1]])

In [3]:
def fit_predict(a_idx, Xb, alpha=100):
    Xa=X(tr[a_idx]); med=Xa[NUM].median()
    Xa=Xa.fillna(med); Xb=Xb.fillna(med)
    sc=StandardScaler().fit(Xa)
    r=Ridge(alpha=alpha).fit(sc.transform(Xa),T[a_idx])
    return softmax(r.predict(sc.transform(Xb)),axis=1)

for alpha in [10,30,100,300]:
    s=[]
    for y in range(2016,2022):
        ia=(tr.Season<y).values; ib=(tr.Season==y).values
        s.append(log_loss(tr[ib].Res,fit_predict(ia,X(tr[ib]),alpha),labels=CLS))
    print('alpha',alpha,'LogLoss médio (2016-2021):',round(np.mean(s),4),np.round(s,4))
prior=[log_loss(tr[tr.Season==y].Res,np.tile(tr[tr.Season<y].Res.value_counts(normalize=True)[CLS].values,((tr.Season==y).sum(),1)),labels=CLS) for y in range(2016,2022)]
print('baseline frequência histórica:',round(np.mean(prior),4))

alpha 10 LogLoss médio (2016-2021): 1.019 [1.0086 1.0744 0.9755 0.9906 1.0328 1.0319]


alpha 30 LogLoss médio (2016-2021): 1.0189 [1.0079 1.0743 0.9757 0.9907 1.0329 1.0318]


alpha 100 LogLoss médio (2016-2021): 1.0187 [1.0063 1.0741 0.9761 0.991  1.0334 1.0313]


alpha 300 LogLoss médio (2016-2021): 1.0187 [1.0049 1.0737 0.9769 0.9919 1.0346 1.0303]
baseline frequência histórica: 1.0517


In [4]:
ALPHA=100
all_idx=np.ones(len(tr),bool)
pred=fit_predict(all_idx,X(te),ALPHA)
pred=np.clip(pred,0.03,0.90); pred/=pred.sum(1,keepdims=True)
sub=pd.DataFrame(pred,columns=CLS); sub.insert(0,'Id',te.Id.values)
sample=pd.read_csv('data/sample_submission.csv')
assert list(sub.columns)==list(sample.columns) and (sub.Id.values==sample.Id.values).all() and len(sub)==1725
assert np.allclose(sub[CLS].sum(1),1)
sub.to_csv('submission.csv',index=False)
print(sub.head()); print(sub[CLS].mean())

     Id         A         D         H
0  3419  0.246908  0.290613  0.462480
1  3420  0.436225  0.292484  0.271292
2  3421  0.174906  0.264916  0.560179
3  3422  0.233704  0.279080  0.487216
4  3423  0.178020  0.245395  0.576585
A    0.265012
D    0.279859
H    0.455128
dtype: float64
